# Threshold — Colab PBR starter (free T4)

Optional cloud path. Local GIMP / Blender stay the default.

**Naming (must match Engine object Name):** `Stone Block` → `textures/stone_block_albedo.png` (+ `_roughness` / `_metalness` / `_normal`).

The pack includes `threshold-asset.json` so `npm run colab:import -- --zip …` can infer the Engine Name.

**Free tier:** NVIDIA T4 when available · session often ≤12 h · idle ~90 min · quotas change. Save or download before you walk away.

Connect this local `.ipynb` in VS Code: install [Google Colab](https://marketplace.visualstudio.com/items?itemName=Google.colab) → Select Kernel → Colab → Auto Connect → pick **T4 GPU** if offered.

Docs: `docs/COLAB_ASSET_PIPELINE.md`

In [ ]:
# 1) Object name → Threshold slug (same rules as src/shared/artNaming.js)
import hashlib, json, os, re, sys, zipfile
from pathlib import Path

OBJECT_NAME = "Stone Block"  # change to your Engine inspector Name

def slugify(name: str) -> str:
    s = re.sub(r"[^a-z0-9]+", "_", (name or "").strip().lower())
    return s.strip("_") or "object"

def pick_out() -> Path:
    candidates = []
    if Path("/content").exists():
        candidates.append(Path("/content/threshold_out"))
    candidates.append(Path.cwd() / "threshold_out")
    for p in candidates:
        try:
            p.mkdir(parents=True, exist_ok=True)
            return p
        except OSError:
            continue
    p = Path("threshold_out")
    p.mkdir(exist_ok=True)
    return p

def family_of(name: str, slug: str) -> str:
    blob = f"{name} {slug}".lower()
    rules = (
        ("wood", ("wood", "plank", "crate", "oak", "pine", "timber")),
        ("metal", ("metal", "steel", "iron", "brass", "pipe", "plate")),
        ("fabric", ("cloth", "fabric", "canvas", "linen", "wool")),
        ("ground", ("dirt", "ground", "soil", "sand", "gravel", "earth")),
    )
    for fam, keys in rules:
        if any(k in blob for k in keys):
            return fam
    return "stone"

SLUG = slugify(OBJECT_NAME)
FAMILY = family_of(OBJECT_NAME, SLUG)
OUT = pick_out()
print(f"Name={OBJECT_NAME!r} slug={SLUG} family={FAMILY}")
print(f"OUT={OUT}")
print(f"Expected: textures/{SLUG}_albedo.png")
print(f"           textures/{SLUG}_roughness.png")
print(f"           textures/{SLUG}_metalness.png")
print(f"           textures/{SLUG}_normal.png")
print(f"           import/{SLUG}.glb  (mesh still from Blender / Engine)")

In [ ]:
# 2) Probe runtime — never required to finish this notebook
gpu, dev_name, vram = False, "CPU (torch not imported)", None
try:
    import torch
    gpu = bool(torch.cuda.is_available())
    if gpu:
        dev_name = torch.cuda.get_device_name(0)
        vram = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
    else:
        dev_name = "CPU (no CUDA)"
except Exception as e:
    dev_name = f"CPU ({e.__class__.__name__})"
print("GPU:", gpu, dev_name, f"~{vram} GB" if vram else "")
print("Honest: free T4 is not guaranteed. If this is CPU-only, skip the SD cell.")

In [ ]:
# 3) Always-works tileable albedo (no model download) — family from OBJECT_NAME
import numpy as np
from PIL import Image

SIZE = 1024
seed = int(hashlib.md5(SLUG.encode("utf-8")).hexdigest()[:8], 16)
rng = np.random.default_rng(seed)
y, x = np.mgrid[0:SIZE, 0:SIZE].astype(np.float32)

def fbm(octaves=5, base=3):
    val = np.zeros((SIZE, SIZE), dtype=np.float32)
    amp = 1.0
    freq = float(base)
    for _ in range(octaves):
        px, py = rng.random() * 2 * np.pi, rng.random() * 2 * np.pi
        val += amp * np.sin(2 * np.pi * freq * x / SIZE + px) * np.cos(2 * np.pi * freq * y / SIZE + py)
        amp *= 0.5
        freq *= 2
    val -= val.min()
    val /= (val.max() + 1e-6)
    return val

PAL = {
    "stone":  np.array([154, 149, 140], np.float32),
    "wood":   np.array([118, 86, 52], np.float32),
    "metal":  np.array([148, 152, 158], np.float32),
    "fabric": np.array([96, 108, 92], np.float32),
    "ground": np.array([132, 108, 78], np.float32),
}
grain = fbm()
vein = 0.07 * np.sin(2 * np.pi * 6 * x / SIZE + grain * 3)
rgb = np.clip(PAL[FAMILY] / 255.0 + (grain[..., None] - 0.5) * 0.22 + vein[..., None], 0, 1)
if FAMILY == "wood":
    rings = 0.08 * np.sin(2 * np.pi * 18 * y / SIZE + grain * 2)
    rgb = np.clip(rgb + rings[..., None] * np.array([0.12, 0.06, 0.02]), 0, 1)
albedo = (rgb * 255).astype(np.uint8)
Image.fromarray(albedo).save(OUT / f"{SLUG}_albedo.png")
print("wrote", OUT / f"{SLUG}_albedo.png", "family", FAMILY)

### Optional — SD 1.5 albedo on T4
Skip if no GPU or you are happy with the seed above. First run downloads weights (~4 GB). Can OOM or hit Colab quota. Keep prompts **material-only** (no character, no text).

In [ ]:
# 4) OPTIONAL diffusion albedo (T4). Set RUN_SD = True only if cell 2 showed a GPU.
RUN_SD = False
SD_PROMPTS = {
    "stone": "seamless tileable weathered stone block albedo, PBR, even lighting, no text, no logo",
    "wood": "seamless tileable oak wood plank albedo, PBR, even lighting, no text, no logo",
    "metal": "seamless tileable brushed steel plate albedo, PBR, even lighting, no text, no logo",
    "fabric": "seamless tileable woven canvas fabric albedo, PBR, even lighting, no text, no logo",
    "ground": "seamless tileable dry dirt gravel ground albedo, PBR, even lighting, no text, no logo",
}
PROMPT = SD_PROMPTS.get(FAMILY, SD_PROMPTS["stone"])

if RUN_SD:
    import torch
    if not torch.cuda.is_available():
        raise SystemExit("No GPU — leave RUN_SD=False and use the procedural albedo.")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "diffusers", "transformers", "accelerate", "safetensors"])
    from diffusers import StableDiffusionPipeline
    pipe = StableDiffusionPipeline.from_pretrained(
        "runwayml/stable-diffusion-v1-5", torch_dtype=torch.float16
    ).to("cuda")
    img = pipe(PROMPT, width=512, height=512, num_inference_steps=20, guidance_scale=7).images[0]
    img = img.resize((SIZE, SIZE), Image.LANCZOS)
    img.save(OUT / f"{SLUG}_albedo.png")
    print("SD albedo wrote", OUT / f"{SLUG}_albedo.png")
    del pipe
    torch.cuda.empty_cache()
else:
    print("Skipped SD (RUN_SD=False). Procedural albedo already on disk.")

In [ ]:
# 5) Derive roughness + metalness + wrap-friendly normal
alb = np.array(Image.open(OUT / f"{SLUG}_albedo.png").convert("RGB")).astype(np.float32) / 255.0
lum = alb @ np.array([0.2126, 0.7152, 0.0722], np.float32)
if FAMILY == "metal":
    rough = np.clip(0.28 + (0.5 - lum) * 0.22, 0.08, 0.55)
    metal = np.clip(0.78 + lum * 0.16, 0.62, 0.96)
else:
    rough = np.clip(0.58 + (0.5 - lum) * 0.28, 0.12, 0.92)
    metal = np.full_like(lum, 0.04)

def save_gray(arr, slot):
    u8 = (np.clip(arr, 0, 1) * 255).astype(np.uint8)
    Image.fromarray(np.stack([u8, u8, u8], -1)).save(OUT / f"{SLUG}_{slot}.png")

save_gray(rough, "roughness")
save_gray(metal, "metalness")

dx = np.roll(lum, -1, axis=1) - np.roll(lum, 1, axis=1)
dy = np.roll(lum, -1, axis=0) - np.roll(lum, 1, axis=0)
nx, ny, nz = -dx * 1.4, -dy * 1.4, np.ones_like(lum)
nlen = np.sqrt(nx * nx + ny * ny + nz * nz)
nx, ny, nz = nx / nlen, ny / nlen, nz / nlen
nmap = np.stack([(nx * 0.5 + 0.5), (ny * 0.5 + 0.5), (nz * 0.5 + 0.5)], -1)
Image.fromarray((nmap * 255).astype(np.uint8)).save(OUT / f"{SLUG}_normal.png")
print("wrote roughness + metalness + normal")

In [ ]:
# 6) Manifest + zip — import can infer Engine Name (no --name required)
manifest = {
    "schema": "threshold-asset/v1",
    "objectName": OBJECT_NAME,
    "slug": SLUG,
    "family": FAMILY,
    "slots": ["albedo", "roughness", "metalness", "normal"],
    "engineNameMustMatch": OBJECT_NAME,
    "importHint": f"npm run colab:import -- --zip {SLUG}_threshold_pbr.zip",
}
(OUT / "threshold-asset.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
zip_path = OUT.parent / f"{SLUG}_threshold_pbr.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for slot in manifest["slots"]:
        p = OUT / f"{SLUG}_{slot}.png"
        z.write(p, p.name)
    z.write(OUT / "threshold-asset.json", "threshold-asset.json")
    z.writestr(
        "README.txt",
        (
            f"Threshold PBR pack for {OBJECT_NAME!r} ({SLUG})\n"
            f"  npm run colab:import -- --zip {zip_path.name}\n"
            f"Engine Name must stay exactly: {OBJECT_NAME}\n"
        ),
    )
print("zip:", zip_path)
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception as e:
    print("Not in Colab browser UI — copy from", zip_path, "—", e)

## After download

1. Save the zip next to the Threshold repo (or anywhere).
2. `npm run colab:import -- --zip path/to/stone_block_threshold_pbr.zip`
   (`--name` is optional when `threshold-asset.json` is in the zip. HILOD runs unless you pass `--no-hilod`.)
3. Keep `npm run textures:watch` running (or reload the Engine). Name the object **exactly** the `OBJECT_NAME` you set.
4. Prefer `MaterialPresets` first; these maps are the hero skin, not CanvasTexture.

This notebook does **not** export a GLB. Mesh still comes from Engine primitives, Blender, or INSERT.